# Llama-3-Taiwan-8B — Hakka Medical Dialogue (四縣腔)

Fine-tune `yentinglin/Llama-3-Taiwan-8B-Instruct` để làm **bác sĩ nói tiếng Hakka (四縣腔)** trong domain y tế.

| Stage | Mô tả |
|-------|-------|
| **1. Data** | `客語漢字QA/` (41 dialogues) + `HAKKA_UTF8_TSV/` (Hakka side, 12 dialogues — dedup) |
| **2. Format** | Sliding window = 6 turns → predict Doctor turn |
| **3. Fine-tune** | LoRA (Unsloth), Doctor role only |
| **4. Evaluation** | BLEU · ROUGE-1/2/L · Perplexity — baseline vs fine-tuned |

**Dataset structure:**
- `客語漢字QA/` — 41 dialogues, 888 turns, Hakka-only (四縣 suy đoán)
- `HAKKA_UTF8_TSV/` — 12 dialogues (subset của QA), có thêm Hakka拼音 & Mandarin
- Sau dedup: **41 unique dialogues**
- Split: 5 dialogues (test) / 36 dialogues (train)

## 1. Install

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import re
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(__import__('torch').__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install -q sentencepiece protobuf "datasets>=2.19" "huggingface_hub>=0.34.0"
    !pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install -q transformers==4.56.2 rouge-score

## 2. Configuration

In [ ]:
import os, random

# ── Paths ──────────────────────────────────────────────────────────────────────
DATASET_ROOT   = "./dataset"          # Colab
# DATASET_ROOT = "../../dataset"       # Local
QA_DIR  = os.path.join(DATASET_ROOT, "客語漢字QA")
TSV_DIR = os.path.join(DATASET_ROOT, "HAKKA_UTF8_TSV")

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID       = "yentinglin/Llama-3-Taiwan-8B-Instruct"
OUTPUT_DIR     = "outputs/hakka_dialogue"
LOAD_IN_4BIT   = True

# ── Dialogue config ───────────────────────────────────────────────────────────
WINDOW_SIZE    = 6      # turns of context before predicting Doctor turn
TEST_N         = 5      # number of dialogues held out for test (1 per topic if possible)
RANDOM_SEED    = 42

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "你係一位客語（四縣腔）醫生，請用純客語（四縣腔）同病人進行醫療問診。"
    "毋好用華語抑係英文回覆，所有回覆愛用客語漢字。"
)

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.05
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj",
                  "gate_proj","up_proj","down_proj"]

# ── Training ──────────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 512
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
MAX_STEPS      = 300
LR             = 2e-4
WARMUP_RATIO   = 0.1

# ── Eval ──────────────────────────────────────────────────────────────────────
EVAL_BATCH_SIZE = 4
MAX_NEW_TOKENS  = 128

random.seed(RANDOM_SEED)
print("Config OK")
print(f"Window size  : {WINDOW_SIZE} turns")
print(f"Test set     : {TEST_N} dialogues (held out)")
print(f"4-bit        : {LOAD_IN_4BIT}")
print(f"Data root    : {DATASET_ROOT}")

## 3. Data Loading

In [ ]:
import glob

def load_qa_dialogues(qa_dir):
    """
    Load 客語漢字QA — mỗi file là 1 dialogue.
    Returns list of dicts: {id, topic, file, turns: [(role, text), ...]}
    Role: 'doctor' (odd rows) | 'patient' (even rows)
    """
    dialogues = []
    for f in sorted(glob.glob(os.path.join(qa_dir, "*.tsv"))):
        with open(f, encoding='utf-8') as fp:
            lines = [l.strip() for l in fp if l.strip() and l.strip() != '客語漢字']
        if not lines:
            continue
        turns = []
        for i, line in enumerate(lines):
            role = 'doctor' if i % 2 == 0 else 'patient'
            turns.append((role, line))
        name = os.path.basename(f).replace('.tsv', '')
        topic = name.rstrip('0123456789')
        dialogues.append({
            'id'    : name,
            'topic' : topic,
            'source': 'QA',
            'turns' : turns,
        })
    return dialogues


def load_tsv_dialogues(tsv_dir):
    """
    Load HAKKA_UTF8_TSV — UTF-16-LE, 6 columns.
    Extract Hakka side only (客語漢字 column).
    Returns same format as load_qa_dialogues.
    """
    dialogues = []
    for f in sorted(glob.glob(os.path.join(tsv_dir, "*.txt"))):
        with open(f, encoding='utf-16-le') as fp:
            content = fp.read()
        lines = [l for l in content.splitlines() if l.strip()]
        turns = []
        for line in lines:
            cols = line.split('\t')
            if len(cols) != 6:
                continue
            audio, speaker, dialect, pinyin, hakka, mandarin = cols
            audio = audio.lstrip('\ufeff')
            if audio == '音訊檔案名稱':  # header
                continue
            role = 'doctor' if speaker.strip() == 'speaker01' else 'patient'
            hakka = hakka.strip()
            if hakka:
                turns.append((role, hakka))
        if not turns:
            continue
        name = os.path.basename(f).replace('.txt', '')
        topic = name.rstrip('0123456789')
        dialogues.append({
            'id'    : name,
            'topic' : topic,
            'source': 'TSV',
            'turns' : turns,
        })
    return dialogues


qa_dialogues  = load_qa_dialogues(QA_DIR)
tsv_dialogues = load_tsv_dialogues(TSV_DIR)

print(f"客語漢字QA : {len(qa_dialogues)} dialogues")
print(f"HAKKA_TSV  : {len(tsv_dialogues)} dialogues")

In [ ]:
from collections import Counter

# ── Dedup: TSV ⊆ QA (12 dialogues overlap) ────────────────────────────────────
# QA already contains the Hakka side of all TSV dialogues → use QA as canonical.
# Only add TSV dialogues whose id is NOT in QA.
qa_ids = {d['id'] for d in qa_dialogues}
extra  = [d for d in tsv_dialogues if d['id'] not in qa_ids]

all_dialogues = qa_dialogues + extra
random.shuffle(all_dialogues)

# ── Stats ─────────────────────────────────────────────────────────────────────
topic_counts = Counter(d['topic'] for d in all_dialogues)
total_turns  = sum(len(d['turns']) for d in all_dialogues)
total_rounds = sum(len(d['turns']) // 2 for d in all_dialogues)

print(f"Total dialogues : {len(all_dialogues)}")
print(f"Total turns     : {total_turns}")
print(f"Total rounds    : {total_rounds}")
print(f"\nBy topic:")
for topic, cnt in sorted(topic_counts.items()):
    print(f"  {topic:<12}: {cnt} dialogues")

## 4. Train / Test Split (by dialogue)

In [ ]:
# ── Hold out 1 dialogue per topic (up to TEST_N total) ────────────────────────
topics = list(topic_counts.keys())
test_ids  = set()
test_pool = {t: [d for d in all_dialogues if d['topic'] == t] for t in topics}

for topic in sorted(test_pool):
    if len(test_ids) >= TEST_N:
        break
    candidates = test_pool[topic]
    if candidates:
        picked = random.choice(candidates)
        test_ids.add(picked['id'])

train_dialogues = [d for d in all_dialogues if d['id'] not in test_ids]
test_dialogues  = [d for d in all_dialogues if d['id'] in test_ids]

print(f"Train : {len(train_dialogues)} dialogues  "
      f"({sum(len(d['turns'])//2 for d in train_dialogues)} rounds)")
print(f"Test  : {len(test_dialogues)} dialogues  "
      f"({sum(len(d['turns'])//2 for d in test_dialogues)} rounds)")
print(f"\nTest dialogues held out:")
for d in test_dialogues:
    print(f"  {d['id']}  ({len(d['turns'])} turns)")

## 5. Build Sliding Window Examples

In [ ]:
def build_examples(dialogues, window_size=WINDOW_SIZE):
    """
    Sliding window: cho mỗi Doctor turn (không phải turn đầu tiên),
    lấy tối đa `window_size` turns trước đó làm context.
    Trả về list of {'context': [(role, text), ...], 'response': str}
    
    context luôn kết thúc bằng Patient turn (để model predict Doctor).
    """
    examples = []
    for d in dialogues:
        turns = d['turns']
        for i, (role, text) in enumerate(turns):
            if role != 'doctor':
                continue
            if i == 0:
                continue  # skip opening greeting (no patient context yet)
            # context = up to window_size turns before i
            start = max(0, i - window_size)
            ctx = turns[start:i]
            # ensure context ends with patient turn
            if not ctx or ctx[-1][0] != 'patient':
                continue
            # drop leading doctor turns so context starts with patient
            while ctx and ctx[0][0] == 'doctor':
                ctx = ctx[1:]
            if not ctx:
                continue
            examples.append({
                'context' : ctx,
                'response': text,
                'topic'   : d['topic'],
                'dial_id' : d['id'],
            })
    return examples


train_examples = build_examples(train_dialogues)
test_examples  = build_examples(test_dialogues)
random.shuffle(train_examples)

print(f"Train examples : {len(train_examples)}")
print(f"Test  examples : {len(test_examples)}")

# Show 1 sample
ex = train_examples[0]
print("\n=== Sample training example ===")
print(f"Topic : {ex['topic']}")
for role, text in ex['context']:
    label = '[Patient]' if role == 'patient' else '[Doctor ]'
    print(f"  {label} {text}")
print(f"  → [Doctor ] {ex['response']}")

## 6. Load Model (Unsloth)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_R,
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    target_modules      = TARGET_MODULES,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",
    random_state        = RANDOM_SEED,
)
print(model.print_trainable_parameters())

## 7. Format & Tokenize

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(tokenizer, chat_template="llama-3")


def example_to_messages(ex):
    """
    Convert a sliding-window example to chat messages.
    context: [(role, text), ...] ending with patient turn
    response: doctor's next turn
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for role, text in ex['context']:
        if role == 'patient':
            messages.append({"role": "user",      "content": text})
        else:
            messages.append({"role": "assistant", "content": text})
    messages.append({"role": "assistant", "content": ex['response']})
    return messages


def format_examples(batch):
    texts = []
    for ex in batch['example']:
        msgs = example_to_messages(ex)
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}


train_ds = Dataset.from_list([{'example': ex} for ex in train_examples])
train_ds = train_ds.map(format_examples, batched=True)

print(f"Train dataset : {len(train_ds):,} examples")
print("\n=== Sample formatted text ===")
print(train_ds[0]['text'][:600], "...")

## 8. Fine-tune (SFTTrainer)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_ds,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    packing            = False,
    args = SFTConfig(
        per_device_train_batch_size   = BATCH_SIZE,
        gradient_accumulation_steps   = GRAD_ACCUM,
        max_steps                     = MAX_STEPS,
        warmup_ratio                  = WARMUP_RATIO,
        learning_rate                 = LR,
        fp16                          = not torch.cuda.is_bf16_supported(),
        bf16                          = torch.cuda.is_bf16_supported(),
        logging_steps                 = 20,
        optim                         = "adamw_8bit",
        weight_decay                  = 0.01,
        lr_scheduler_type             = "cosine",
        seed                          = RANDOM_SEED,
        output_dir                    = OUTPUT_DIR,
        report_to                     = "none",
    ),
)

trainer_stats = trainer.train()
print(f"\nTraining done. Steps: {trainer_stats.global_step}  "
      f"Loss: {trainer_stats.training_loss:.4f}")

## 9. Evaluation Functions

- **BLEU**: character-level n-gram (reused from translation notebook)
- **ROUGE-1/2/L**: character-level overlap
- **Perplexity (PPL)**: model uncertainty on test Doctor turns

In [ ]:
# ── BLEU (reused from Hakka Translation notebook) ─────────────────────────────
import math
from collections import Counter
from typing import List, Sequence

def _norm(text): return "".join((text or "").strip().split())
def _tok(text):  return list(_norm(text))

def _ngrams(tokens: Sequence[str], n: int) -> Counter:
    if len(tokens) < n: return Counter()
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def sentence_bleu(reference: str, hypothesis: str, max_n: int = 4, smooth: float = 1.0) -> float:
    ref, hyp = _tok(reference), _tok(hypothesis)
    if not hyp: return 0.0
    precs = []
    for n in range(1, max_n+1):
        rc, hc = _ngrams(ref, n), _ngrams(hyp, n)
        clip = total = 0
        for ng, cnt in hc.items():
            clip  += min(cnt, rc.get(ng, 0))
            total += cnt
        precs.append((clip+smooth)/(total+smooth) if total else 0.0)
    if min(precs) <= 0: return 0.0
    geo = math.exp(sum(math.log(p) for p in precs) / max_n)
    bp  = 1.0 if len(hyp) >= len(ref) else math.exp(1.0 - len(ref)/max(len(hyp),1))
    return bp * geo

def corpus_bleu(references: List[str], hypotheses: List[str], max_n: int = 4) -> float:
    if len(references) != len(hypotheses) or not references: return 0.0
    clip_tot = [0]*max_n; hyp_tot = [0]*max_n
    ref_len = hyp_len = 0
    for ref, hyp in zip(references, hypotheses):
        rt, ht = _tok(ref), _tok(hyp)
        ref_len += len(rt); hyp_len += len(ht)
        for n in range(1, max_n+1):
            rc, hc = _ngrams(rt, n), _ngrams(ht, n)
            for ng, cnt in hc.items():
                clip_tot[n-1] += min(cnt, rc.get(ng, 0))
            hyp_tot[n-1] += len(ht) - n + 1 if len(ht) >= n else 0
    precs = []
    for n in range(max_n):
        precs.append(clip_tot[n]/max(hyp_tot[n],1))
    if min(precs) <= 0: return 0.0
    geo = math.exp(sum(math.log(max(p,1e-9)) for p in precs) / max_n)
    bp  = 1.0 if hyp_len >= ref_len else math.exp(1.0 - ref_len/max(hyp_len,1))
    return bp * geo


# ── ROUGE (character-level) ───────────────────────────────────────────────────
def _lcs_len(x, y):
    """LCS length via DP."""
    m, n = len(x), len(y)
    if m == 0 or n == 0: return 0
    dp = [[0]*(n+1) for _ in range(2)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i%2][j] = dp[(i-1)%2][j-1]+1 if x[i-1]==y[j-1] else max(dp[(i-1)%2][j], dp[i%2][j-1])
    return dp[m%2][n]

def rouge_scores(reference: str, hypothesis: str) -> dict:
    """Compute ROUGE-1, ROUGE-2, ROUGE-L (character-level)."""
    ref, hyp = _tok(reference), _tok(hypothesis)
    if not ref or not hyp:
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}

    def f1(p, r): return 2*p*r/(p+r) if (p+r) > 0 else 0.0

    # ROUGE-1
    rc1, hc1 = Counter(ref), Counter(hyp)
    hit1 = sum(min(rc1[c], hc1[c]) for c in hc1)
    r1 = f1(hit1/len(hyp), hit1/len(ref))

    # ROUGE-2
    rc2, hc2 = _ngrams(ref, 2), _ngrams(hyp, 2)
    hit2 = sum(min(rc2[g], hc2[g]) for g in hc2)
    denom_r2 = max(len(hyp)-1, 1); denom_ref2 = max(len(ref)-1, 1)
    r2 = f1(hit2/denom_r2, hit2/denom_ref2)

    # ROUGE-L
    lcs = _lcs_len(ref, hyp)
    rL = f1(lcs/len(hyp), lcs/len(ref))

    return {'rouge1': r1, 'rouge2': r2, 'rougeL': rL}


# ── Generation helper ─────────────────────────────────────────────────────────
def generate_doctor_response(context, model, tokenizer, max_new_tokens=MAX_NEW_TOKENS):
    """Given context [(role,text),...] ending with patient, generate Doctor reply."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for role, text in context:
        r = "user" if role == 'patient' else "assistant"
        messages.append({"role": r, "content": text})
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            temperature    = 1.0,
            pad_token_id   = tokenizer.eos_token_id,
        )
    gen_ids = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


# ── Perplexity helper ─────────────────────────────────────────────────────────
def compute_ppl(examples, model, tokenizer):
    """Average PPL on the Doctor response tokens of each example."""
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    with torch.no_grad():
        for ex in examples:
            msgs = example_to_messages(ex)
            text = tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            )
            ids = tokenizer(text, return_tensors="pt")["input_ids"].to(model.device)
            if ids.shape[1] > MAX_SEQ_LENGTH:
                continue
            out = model(ids, labels=ids)
            nll = out.loss.item()
            total_nll    += nll * (ids.shape[1] - 1)
            total_tokens += (ids.shape[1] - 1)
    ppl = math.exp(total_nll / max(total_tokens, 1))
    return round(ppl, 4)


print("Evaluation functions loaded.")

## 10. Baseline Evaluation (Zero-shot)

Đánh giá **base model chưa fine-tune** trên test set với cùng prompt.

In [ ]:
from unsloth import FastLanguageModel as FLM

# Load base model (no LoRA, inference mode)
base_model, base_tokenizer = FLM.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)
base_tokenizer = get_chat_template(base_tokenizer, chat_template="llama-3")
FLM.for_inference(base_model)
print("Base model loaded for inference.")

In [ ]:
def run_eval(model, tok, examples, label):
    """
    Run BLEU + ROUGE + PPL evaluation.
    Returns summary dict + detailed rows.
    """
    print(f"\n{'─'*55}")
    print(f"Evaluating: {label}  ({len(examples)} examples)")
    print(f"{'─'*55}")

    refs_all, hyps_all = [], []
    rows = []
    bleus, r1s, r2s, rLs = [], [], [], []

    for i, ex in enumerate(examples):
        ref  = ex['response']
        pred = generate_doctor_response(ex['context'], model, tok)

        bleu = sentence_bleu(ref, pred)
        rg   = rouge_scores(ref, pred)

        bleus.append(bleu)
        r1s.append(rg['rouge1'])
        r2s.append(rg['rouge2'])
        rLs.append(rg['rougeL'])
        refs_all.append(ref)
        hyps_all.append(pred)
        rows.append({
            'topic': ex['topic'],
            'reference' : ref,
            'prediction': pred,
            'bleu' : round(bleu, 4),
            'rouge1': round(rg['rouge1'], 4),
            'rouge2': round(rg['rouge2'], 4),
            'rougeL': round(rg['rougeL'], 4),
        })
        if (i+1) % 10 == 0:
            print(f"  [{i+1}/{len(examples)}] avg BLEU={sum(bleus)/len(bleus):.4f}")

    c_bleu = corpus_bleu(refs_all, hyps_all)
    ppl    = compute_ppl(examples, model, tok)

    summary = {
        'label'        : label,
        'n'            : len(examples),
        'avg_bleu'     : round(sum(bleus)/len(bleus), 4),
        'corpus_bleu'  : round(c_bleu, 4),
        'avg_rouge1'   : round(sum(r1s)/len(r1s), 4),
        'avg_rouge2'   : round(sum(r2s)/len(r2s), 4),
        'avg_rougeL'   : round(sum(rLs)/len(rLs), 4),
        'perplexity'   : ppl,
    }
    print(f"\n  avg BLEU     = {summary['avg_bleu']:.4f}")
    print(f"  corpus BLEU  = {summary['corpus_bleu']:.4f}")
    print(f"  avg ROUGE-1  = {summary['avg_rouge1']:.4f}")
    print(f"  avg ROUGE-2  = {summary['avg_rouge2']:.4f}")
    print(f"  avg ROUGE-L  = {summary['avg_rougeL']:.4f}")
    print(f"  Perplexity   = {summary['perplexity']:.2f}")
    return summary, rows


baseline_summary, baseline_rows = run_eval(
    base_model, base_tokenizer, test_examples, label="Baseline (zero-shot)"
)

## 11. Fine-tuned Model Evaluation

In [ ]:
FastLanguageModel.for_inference(model)  # enable fast inference mode

ft_summary, ft_rows = run_eval(
    model, tokenizer, test_examples, label="Fine-tuned (LoRA)"
)

## 12. Comparison Summary

In [ ]:
def delta(a, b, higher_better=True):
    diff = b - a
    sign = '+' if diff >= 0 else ''
    arrow = '▲' if (diff > 0) == higher_better else ('▼' if diff != 0 else '─')
    return f"{sign}{diff:.4f} {arrow}"

bs, ft = baseline_summary, ft_summary

print("\n" + "═"*72)
print(f"{'Metric':<18} {'Baseline':>12} {'Fine-tuned':>12} {'Delta':>18}")
print("─"*72)
metrics = [
    ('avg BLEU',    'avg_bleu',    True),
    ('corpus BLEU', 'corpus_bleu', True),
    ('ROUGE-1',     'avg_rouge1',  True),
    ('ROUGE-2',     'avg_rouge2',  True),
    ('ROUGE-L',     'avg_rougeL',  True),
    ('Perplexity',  'perplexity',  False),  # lower = better
]
for name, key, hb in metrics:
    bv = bs[key]; fv = ft[key]
    print(f"{name:<18} {bv:>12.4f} {fv:>12.4f} {delta(bv, fv, hb):>18}")
print("═"*72)
print(f"Test examples  : {ft['n']}")
print(f"Model          : {MODEL_ID}")
print(f"Window size    : {WINDOW_SIZE} turns")

In [ ]:
# ── Qualitative comparison: same context, both models ─────────────────────────
print("=== Qualitative comparison (5 examples) ===")
for i, ex in enumerate(test_examples[:5]):
    print(f"\n[{i+1}] Topic: {ex['topic']}")
    print("Context:")
    for role, text in ex['context'][-2:]:  # show last 2 turns
        label = 'Patient' if role == 'patient' else 'Doctor '
        print(f"  [{label}] {text}")
    print(f"Reference : {ex['response']}")
    print(f"Baseline  : {baseline_rows[i]['prediction']}")
    print(f"Fine-tuned: {ft_rows[i]['prediction']}")
    print(f"BLEU  baseline={baseline_rows[i]['bleu']:.4f}  ft={ft_rows[i]['bleu']:.4f}")

## 13. Save Model

In [ ]:
# ── Save LoRA adapters ────────────────────────────────────────────────────────
SAVE_PATH = os.path.join(OUTPUT_DIR, "lora_adapters")
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"LoRA adapters saved → {SAVE_PATH}")

# ── Optional: merge & save full model ────────────────────────────────────────
# merged = model.merge_and_unload()
# merged.save_pretrained(os.path.join(OUTPUT_DIR, "merged"))
# tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "merged"))

# ── Optional: push to HuggingFace Hub ────────────────────────────────────────
# model.push_to_hub("your-username/hakka-dialogue-doctor", token="hf_...")

## 14. Inference Demo

Thử chat với model — nhập câu của bệnh nhân bằng tiếng Hakka.

In [ ]:
def chat_demo(patient_turns, model, tokenizer):
    """
    patient_turns: list of patient Hakka utterances (simulating conversation history)
    Returns doctor's response.
    """
    context = [('patient', t) for t in patient_turns]
    resp = generate_doctor_response(context, model, tokenizer)
    return resp


# Example 1: 腎臟泌尿系統 (kidney/urinary)
history_1 = [
    "先生，𠊎這駁仔密密愛屙尿，日時頭差毋多半點鐘就愛走一到便所。"
]
print("=== Demo 1: 泌尿 ===" )
print(f"Patient : {history_1[-1]}")
print(f"Doctor  : {chat_demo(history_1, model, tokenizer)}")

print()

# Example 2: 全身性症狀 (systemic)
history_2 = [
    "先生，𠊎對昨暗晡開始作燒，直直燒到今，大約有三十八點五度。"
]
print("=== Demo 2: 發燒 ===")
print(f"Patient : {history_2[-1]}")
print(f"Doctor  : {chat_demo(history_2, model, tokenizer)}")